In [11]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_energy_usage.csv to cleaned_energy_usage.csv


In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SmartHomeEnergy").getOrCreate()

df = spark.read.csv("cleaned_energy_usage.csv", header=True, inferSchema=True)

df = df.withColumn("hour", F.hour("timestamp"))
is_peak = (F.col("hour") >= 18) & (F.col("hour") < 22)

device_usage = df.groupBy("device_id").agg(
    F.sum(F.when(is_peak, F.col("energy_kwh")).otherwise(0)).alias("peak_kwh"),
    F.sum(F.when(~is_peak, F.col("energy_kwh")).otherwise(0)).alias("off_peak_kwh"),
    F.sum("energy_kwh").alias("total_kwh")
)

device_usage.show()

top_devices = device_usage.orderBy(F.desc("total_kwh")).limit(5)
top_devices.show()

top_devices.write.csv("top_devices_output", header=True, mode="overwrite")

spark.stop()

+---------+------------------+------------------+------------------+
|device_id|          peak_kwh|      off_peak_kwh|         total_kwh|
+---------+------------------+------------------+------------------+
|        1|16.300999999999956|2160.8739999999966|2177.1749999999965|
|        6| 94.89600000000002|229.02399999999994| 323.9199999999999|
|        3| 20.02300000000004|148.77100000000152| 168.7939999999996|
|        5| 89.48900000000015|444.03999999999775| 533.5289999999975|
|        9| 9.651999999999905| 48.35900000000098| 58.01100000000159|
|        4|62.876999999999796|113.78400000000212|176.66099999999813|
|        8|           522.092| 706.4440000000006|1228.5359999999982|
|        7|               0.0|176.89299999999986|176.89299999999986|
|       10| 32.45299999999983|23.628000000000455| 56.08099999999941|
|        2| 128.4629999999995| 82.77100000000057| 211.2339999999951|
+---------+------------------+------------------+------------------+

+---------+------------------+---